# Capacity-constrained spatial clustering and cluster-wise balancing

This notebook creates the final labeled sample dataset for CNN/SSL landslide susceptibility modeling under spatial cross-validation. Ordinary KMeans is used only to initialize landslide spatial centroids. Final landslide cluster labels are produced by capacity-constrained assignment so the positive samples are split into nearly equal folds.

## 1. Load paths and parameters

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.spatial_cluster_balance import (
    ClusterBalanceConfig,
    FACTOR_COLUMNS,
    assign_negatives_to_balanced_centroids,
    build_balanced_capacities,
    build_final_cluster_balanced_dataset,
    capacity_constrained_landslide_assignment,
    cluster_wise_balance,
    load_landslide_samples,
    load_reliable_nonlandslide_samples,
    save_final_dataset,
    search_kmeans_landslide_clusters,
)

config = ClusterBalanceConfig(
    project_root=PROJECT_ROOT,
    landslide_points_csv=PROJECT_ROOT / "data/raw/samples/landslide samples.csv",
    reliable_nonlandslide_csv=PROJECT_ROOT / "data/processed/pu_bagging/reliable_nonlandslide_points_all.csv",
    cleaned_raster_dir=PROJECT_ROOT / "data/processed/rasters_cleaned",
    output_csv=PROJECT_ROOT / "data/processed/samples/final_cluster_balanced_dataset.csv",
    n_clusters=5,
    random_seed=42,
    kmeans_seed_start=0,
    kmeans_seed_end=100,
    min_landslide_per_cluster=30,
    max_imbalance_ratio=3.0,
).resolve()

print(f"Landslide points: {config.landslide_points_csv}")
print(f"Reliable non-landslide points: {config.reliable_nonlandslide_csv}")
print(f"Cleaned rasters: {config.cleaned_raster_dir}")
print(f"Final output CSV: {config.output_csv}")

## 2. Load landslide and reliable non-landslide samples

In [ ]:
landslides = load_landslide_samples(
    config.landslide_points_csv,
    config.cleaned_raster_dir,
    FACTOR_COLUMNS,
)
reliable_negatives = load_reliable_nonlandslide_samples(
    config.reliable_nonlandslide_csv,
    FACTOR_COLUMNS,
)

print(f"Valid landslide samples: {len(landslides)}")
print(f"Valid reliable non-landslide samples: {len(reliable_negatives)}")

## 3. KMeans centroid initialization on landslides only

In [ ]:
search_result = search_kmeans_landslide_clusters(
    landslide_samples=landslides,
    n_clusters=config.n_clusters,
    kmeans_seed_start=config.kmeans_seed_start,
    kmeans_seed_end=config.kmeans_seed_end,
    min_landslide_per_cluster=config.min_landslide_per_cluster,
    max_imbalance_ratio=config.max_imbalance_ratio,
)

print(f"selected_initialization_random_state: {search_result.selected_random_state}")
print(f"initialization_inertia: {search_result.inertia:.6f}")
print(f"ordinary_kmeans_diagnostic_counts: {search_result.cluster_counts}")

## 4. Capacity-constrained landslide assignment

In [ ]:
target_capacities = build_balanced_capacities(len(landslides), config.n_clusters)
clustered_landslides, balanced_centroids = capacity_constrained_landslide_assignment(
    landslides,
    search_result.kmeans.cluster_centers_,
    target_capacities,
)

final_landslide_counts = clustered_landslides.groupby("cluster_id").size().reindex(range(config.n_clusters), fill_value=0).astype(int)
final_imbalance_ratio = final_landslide_counts.max() / final_landslide_counts.min()
print("clustering method: capacity-constrained KMeans assignment")
print(f"n_landslide: {len(landslides)}")
print(f"n_clusters: {config.n_clusters}")
print(f"target capacities: {target_capacities}")
print(f"final landslide counts by cluster: {final_landslide_counts.to_dict()}")
print(f"final imbalance ratio: {final_imbalance_ratio:.6f}")

## 5. Assign reliable non-landslides to balanced centroids

In [ ]:
clustered_negatives = assign_negatives_to_balanced_centroids(
    reliable_negatives,
    balanced_centroids,
)
negative_available = clustered_negatives.groupby("cluster_id").size().reindex(range(config.n_clusters), fill_value=0).astype(int)
print(f"n_reliable_negative_available by cluster: {negative_available.to_dict()}")

## 6. Cluster-wise 1:1 positive-negative balancing

In [ ]:
selected_negatives, balance_summary = cluster_wise_balance(
    clustered_landslides,
    clustered_negatives,
    n_clusters=config.n_clusters,
    random_seed=config.random_seed,
)

print(balance_summary.to_string(index=False))

## 7. Build and save final labeled dataset

In [ ]:
final_dataset = build_final_cluster_balanced_dataset(
    clustered_landslides,
    selected_negatives,
    n_clusters=config.n_clusters,
    factor_columns=FACTOR_COLUMNS,
)
saved_csv = save_final_dataset(
    final_dataset,
    config.output_csv,
    config.n_clusters,
    FACTOR_COLUMNS,
)

n_landslide = int((final_dataset["label"] == 1).sum())
n_negative = int((final_dataset["label"] == 0).sum())
print("Final dataset summary")
print("clustering method: capacity-constrained KMeans assignment")
print(f"n_landslide: {len(landslides)}")
print(f"n_clusters: {config.n_clusters}")
print(f"target capacities: {target_capacities}")
print(f"final landslide counts by cluster: {final_landslide_counts.to_dict()}")
print(f"final imbalance ratio: {final_imbalance_ratio:.6f}")
print(f"n_reliable_negative_available by cluster: {negative_available.to_dict()}")
print(f"n_negative_selected by cluster: {balance_summary.set_index('cluster_id')['n_negative_selected'].to_dict()}")
print(f"total landslide samples: {n_landslide}")
print(f"total selected non-landslide samples: {n_negative}")
print(f"total samples: {len(final_dataset)}")
print(f"saved output path: {saved_csv}")